## Synthetic data for ASQA

Since HotpotQA provides short-form answers, we apply an answer verbalization step to transform entity-based answers into complete natural language sentences, ensuring compatibility with LLM-based evaluation metrics such as RAGAs.

In [9]:
import pandas as pd
import numpy as np
import os
import dotenv
from typing import Any, Dict, List, Optional
import time
import ast
import re
from google import genai
import google.genai
from google.genai import types
from openai import OpenAI
import asyncio
from asyncio import Semaphore
from tqdm.auto import tqdm  # Auto-select best version (notebook or console)
from dotenv import load_dotenv
from datasets import load_dataset

load_dotenv()

True

## 1. Configuration and Setup

In [57]:
ASQA_PATH = "../data/asqa/train.csv"
ASQA_HALLU_PATH = "../data/asqa/hallu_asqa.csv"
ASQA_LABELED_PATH = "../data/asqa/labeled_asqa.csv"
ASQA_CHECKPOINT_PATH = "../results/synthetic-data/asqa_checkpoint.csv"


WIKI_PATH = "../data/wikiEval/wikieval.csv"
WIKI_LABELED_PATH = "../data/wikiEval/labeled_wikieval.csv"
WIKI_DEV_PATH = "../data/wikiEval/dev_wikieval.csv"


# Output paths
MS_PATH = "../data/ms-marco/msmarco_500.csv"       # keep user's requested filename
MS_HALLU_PATH = "../data/ms-marco/hallu_msmarco.csv"
MS_LABELED_PATH = "../data/ms-marco/labeled_msmarco.csv"
MS_CHECKPOINT_PATH = "../results/synthetic-data/msmarco_checkpoint.csv"

N_SAMPLES = 500

# Context parsing
SELECTED_ONLY = True      # Prefer gold/selected passages when available
MAX_PASSAGES = 10         # Safety cap if fallback to all passages
MIN_ANSWER_WORDS = 5      # Remove empty/noisy answers
MIN_CONTEXT_WORDS = 20    # Remove samples with unusable context

In [8]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# client = genai.Client(api_key=GOOGLE_API_KEY)
client = OpenAI(api_key=OPENAI_API_KEY)
SLEEP_TIME = 1
MODEL_NAME = "gpt-4o-mini"

In [4]:
for m in client.models.list():
    print(m.id)

text-embedding-ada-002
whisper-1
gpt-3.5-turbo
tts-1
gpt-3.5-turbo-16k
gpt-4-0613
gpt-4
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
omni-moderation-latest
omni-moderation-2024-09-26
o1-2024-12-17
o1
o3-mini
o3-mini-2025-01-31
gpt-4o-2024-11-20
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-mini-search-preview
gpt-4o-transcribe
gpt-4o-mini-transcribe
o1-pro-2025-03-19
o1-pro
gpt-4o-mini-tts
o3-2025-04-16
o4-mini-2025-04-16
o3
o4-mini
gpt-4.1-2025-04-14
gpt-4.1
gpt-4.1-mini-2025-04-14
gpt-4.1-mini
gpt-4.1-nano-2025-04-14
gpt-4.1-nano
gpt-image-1
gpt-4o-transcribe-diarize
gpt-5-chat-latest
gpt-5-2025-08-07
gpt-5
gpt-5-mini-2025-08-07
gpt-5-mini
gpt-5-nano-2025-08-07
gpt-5-nano
gpt-audio-2025-08-28
gpt-realtime
gpt-realtime-2025-08-28

In [ ]:
if not os.path.exists("../data/ms-marco"):
    os.makedirs("../data/ms-marco")

if not os.path.exists("../data/wikiEval"):
    os.makedirs("../data/wikiEval")

## 2. Data Loading and Preprocessing

In [6]:
# load dataset
asqa = pd.read_csv(ASQA_PATH)

wiki = load_dataset("vibrantlabsai/WikiEval")
wiki = wiki["train"]
print(wiki)
print(wiki.column_names)

ms = load_dataset("microsoft/ms_marco", "v1.1")
ms = ms["validation"]
print(ms)
print(ms.column_names)

Dataset({
    features: ['answer', 'question', 'context_v1', 'context_v2', 'ungrounded_answer', 'source', 'poor_answer'],
    num_rows: 50
})
['answer', 'question', 'context_v1', 'context_v2', 'ungrounded_answer', 'source', 'poor_answer']


c:\Users\vnpq2\anaconda3\envs\ragas-env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vnpq2\.cache\huggingface\hub\datasets--microsoft--ms_marco. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 9650/9650 [00:00<00:00, 85448.23 examples/s]


Dataset({
    features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
    num_rows: 10047
})
['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers']


### ASQA

In [96]:
def parse_supporting_facts(supporting_facts_raw, context_raw):
    try:
        # ===== clean string =====
        sf_clean = supporting_facts_raw.replace('""', '"')
        ctx_clean = context_raw.replace('""', '"')

        sf = ast.literal_eval(sf_clean)
        ctx = ast.literal_eval(ctx_clean)

        # ===== lấy danh sách title cần dùng =====
        sf_titles = set(sf.get("title", []))

        ctx_titles = ctx.get("title", [])
        ctx_sentences = ctx.get("sentences", [])

        facts = []

        # ===== lọc đúng doc theo title =====
        for title, sents in zip(ctx_titles, ctx_sentences):
            if title in sf_titles or str(title).startswith("QA"):
                joined_sents = " ".join(sents)
                
                facts.append(f"- ({title}) {joined_sents}")

        return "\n".join(facts)

    except Exception as e:
        return ""

In [99]:
asqa['context'] = asqa.apply(lambda row: parse_supporting_facts(row['supporting_facts'], row['context']), axis=1)
display(asqa.head(3))

,id,question,answer,context,supporting_facts
0,asqa_0,When does the new bunk'd come out?,The new bunk'd episode 41 comes out on April 2...,- (List of Bunk'd episodes) The new bunk'd epi...,"{'title': [""List of Bunk'd episodes""]}"
1,asqa_1,Who won the 2016 ncaa football national champi...,The 2015 - 2016 season's ncaa national footbal...,- (2016 College Football Playoff National Cham...,{'title': ['2016 College Football Playoff Nati...
2,asqa_2,When was the last time the death penalty was u...,The last time the death penalty was used in pa...,"- (QA_1) As of 2017, when was the last time th...",{'title': ['Gary M. Heidnik']}


In [102]:
# rename columns "answer" -> "gold_answer"
asqa = asqa.rename(columns={"answer": "gold_answer"})

### MS Macro

In [61]:
def normalize_text(x: Any) -> str:
    """Convert MS MARCO field values into clean plain text."""
    if x is None:
        return ""

    if isinstance(x, str):
        return re.sub(r"\s+", " ", x).strip()

    if isinstance(x, (list, tuple, np.ndarray)):
        parts = [normalize_text(v) for v in x]
        parts = [p for p in parts if p and p.lower() not in {"no answer present.", "no answer present"}]
        return " ".join(parts).strip()

    return re.sub(r"\s+", " ", str(x)).strip()


def pick_gold_answer(row: Dict[str, Any]) -> str:
    """Pick the longest available answer for a sample.

    MS MARCO usually stores answers as a list in `answers`.
    We choose the longest non-empty answer because this notebook focuses on long-form samples.
    """
    answers = row.get("answers", row.get("answer", ""))

    if isinstance(answers, (list, tuple, np.ndarray)):
        candidates = [
            normalize_text(a)
            for a in answers
            if normalize_text(a)
            and normalize_text(a).lower() not in {"no answer present.", "no answer present"}
        ]
        if not candidates:
            return ""
        return max(candidates, key=lambda s: len(s.split()))

    answer = normalize_text(answers)
    if answer.lower() in {"no answer present.", "no answer present"}:
        return ""
    return answer


def _get_passage_list(passages: Any) -> List[Dict[str, Any]]:
    """Normalize MS MARCO passages into a list of dicts.

    Expected MS MARCO HF structure is usually:
    {
        "is_selected": [...],
        "passage_text": [...],
        "url": [...]
    }

    This function also handles already-list-like formats defensively.
    """
    if passages is None:
        return []

    # HF common case: dict of lists
    if isinstance(passages, dict):
        texts = passages.get("passage_text", passages.get("text", []))
        selected = passages.get("is_selected", [None] * len(texts))
        urls = passages.get("url", [""] * len(texts))

        result = []
        for i, text in enumerate(texts):
            result.append({
                "passage_text": normalize_text(text),
                "is_selected": selected[i] if i < len(selected) else None,
                "url": urls[i] if i < len(urls) else "",
            })
        return result

    # Defensive case: list of dicts or list of strings
    if isinstance(passages, list):
        result = []
        for p in passages:
            if isinstance(p, dict):
                result.append({
                    "passage_text": normalize_text(p.get("passage_text", p.get("text", ""))),
                    "is_selected": p.get("is_selected", p.get("selected", None)),
                    "url": p.get("url", ""),
                })
            else:
                result.append({
                    "passage_text": normalize_text(p),
                    "is_selected": None,
                    "url": "",
                })
        return result

    return [{"passage_text": normalize_text(passages), "is_selected": None, "url": ""}]


def parse_context(passages: Any, selected_only: bool = True, max_passages: int = 10) -> str:
    """Build a clean gold context string.

    Design:
    1. Prefer passages marked `is_selected == 1` because they are closer to gold context.
    2. If no selected passage exists, fallback to the first `max_passages` non-empty passages.
    3. Keep passage boundaries with [P1], [P2], ... so the model can reason over evidence.
    """
    passage_list = _get_passage_list(passages)
    passage_list = [p for p in passage_list if p["passage_text"]]

    if selected_only:
        selected_passages = [
            p for p in passage_list
            if str(p.get("is_selected", "")).lower() in {"1", "true", "yes"}
        ]
    else:
        selected_passages = []

    chosen = selected_passages if selected_passages else passage_list[:max_passages]

    context_blocks = []
    for i, p in enumerate(chosen, start=1):
        url = normalize_text(p.get("url", ""))
        text = p["passage_text"]

        if url:
            context_blocks.append(f"[P{i}] Source: {url}\\n{text} ")
        else:
            context_blocks.append(f"[P{i}] {text}")

    return "\\n\\n".join(context_blocks).strip()


In [62]:
def convert_ms_marco_to_rag_df(dataset) -> pd.DataFrame:
    records = []

    for idx, row in enumerate(tqdm(dataset, desc="Parsing MS MARCO")):
        row = dict(row)

        sample_id = normalize_text(
            row.get("query_id", row.get("id", idx))
        )
        if not sample_id:
            sample_id = str(idx)

        question = normalize_text(
            row.get("query", row.get("question", ""))
        )

        gold_answer = pick_gold_answer(row)
        context = parse_context(
            row.get("passages", row.get("context", "")),
            selected_only=SELECTED_ONLY,
            max_passages=MAX_PASSAGES,
        )

        records.append({
            "id": str(sample_id),
            "question": question,
            "context": context,
            "gold_answer": gold_answer,
        })

    df = pd.DataFrame(records)

    df["answer_len"] = df["gold_answer"].fillna("").str.split().str.len()
    df["context_len"] = df["context"].fillna("").str.split().str.len()

    df = df[
        df["question"].notnull() &
        df["context"].notnull() &
        df["gold_answer"].notnull() &
        (df["question"].str.len() > 0) &
        (df["context_len"] >= MIN_CONTEXT_WORDS) &
        (df["answer_len"] >= MIN_ANSWER_WORDS)
    ].copy()

    return df


msmarco_df = convert_ms_marco_to_rag_df(ms)
print(msmarco_df.shape)
msmarco_df.head()


Parsing MS MARCO: 100%|██████████| 10047/10047 [00:02<00:00, 3431.38it/s]


(6089, 6)


,id,question,context,gold_answer,answer_len,context_len
1,9653,how much do bartenders make,[P1] Source: http://www.breakintobartending.co...,The average hourly wage for a bartender is $10...,16,78
2,9654,what is a furuncle boil,[P1] Source: https://en.wikipedia.org/wiki/Boi...,"A boil, also called a furuncle, is a deep foll...",26,99
3,9655,what can urinalysis detect,[P1] Source: http://www.mayoclinic.org/tests-p...,"Detect and assess a wide range of disorders, s...",17,81
4,9656,what is vitamin a used for,[P1] Source: http://www.webmd.com/vitamins-sup...,"Shigellosis, diseases of the nervous system, n...",32,77
5,9657,what causes genetic alterations in normal cells,[P1] Source: http://www.hindawi.com/journals/i...,The initiation of cell transformation is gener...,26,70


In [63]:
# Pick 500 samples with the longest gold answers
msmarco_500 = (
    msmarco_df
    .sort_values(["answer_len", "context_len"], ascending=False)
    .head(N_SAMPLES)
    .reset_index(drop=True)
)

# Keep only required columns in the exported clean dataset
msmarco_500_export = msmarco_500[["id", "question", "context", "gold_answer"]].copy()
msmarco_500_export.to_csv(MS_PATH, index=False, encoding="utf-8-sig")

print(msmarco_500_export.shape)
print(f"Saved: {MS_PATH}")
msmarco_500_export.head()


(500, 4)
Saved: ../data/ms-marco/msmarco_500.csv


,id,question,context,gold_answer
0,10736,buying a car dealer vs. private seller,[P1] Source: http://www.consumerhelp.ie/buying...,Traders are often called “dealers” and sell ca...
1,12719,convert dollar pay into salary to twice a month,[P1] Source: http://www.ehow.com/how_12044390_...,Determine what your paydays will be throughout...
2,17806,how to plant dwarf apple trees,[P1] Source: http://www.gardenguides.com/79766...,Locate a site for the dwarf apple tree where i...
3,14675,what arthritis is,[P1] Source: http://www.healthline.com/health/...,Arthritis is inflammation of the joints (the p...
4,16014,how to record audio with iphone,[P1] Source: http://www.solveyourtech.com/reco...,1: Open the Voice Memos app. If you cannot fin...


## 3. Prompt Design and LLM Interaction

In [64]:
PROMPT_SYSTEM = """You are an AI that generates hallucinated answers for evaluating contextual faithfulness.

Follow all instructions strictly:
- The answer MUST NOT be fully supported by the provided context.
- The answer should directly conflict with, exaggerate, or add 1-2 unsupported key claims relative to the context.
- The answer should remain fluent, natural, and confident.
- Keep the style and length similar to the gold answer.
- Do NOT mention uncertainty.
- Do NOT say the answer is hallucinated, unsupported, false, or incorrect.
- Do NOT use special formatting, markdown, or bullet points.
- Output plain text only.

The goal is contextual unfaithfulness, not random nonsense.
"""

PROMPT_USER = """Question:
{question}

Gold Answer:
{gold_answer}

Context:
{context}

Return ONLY the hallucinated answer.
"""


In [100]:
def synthetic(row: pd.Series, client: OpenAI, model: str = MODEL_NAME) -> Optional[str]:
    prompt_user = PROMPT_USER.format(
        question=str(row["question"]).strip(),
        gold_answer=str(row["gold_answer"]).strip(),
        context=str(row["context"]).strip(),
    )

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": PROMPT_SYSTEM},
                {"role": "user", "content": prompt_user},
            ],
            temperature=0.7,
        )

        text = response.choices[0].message.content.strip()

        # Basic validation
        if not text:
            return None

        if text.strip().lower() == str(row["gold_answer"]).strip().lower():
            return None

        return text

    except Exception as e:
        print(f"OpenAI error for id={row.get('id', 'unknown')}: {e}")
        return None


## 4. Test on 1 sample

In [103]:
row = asqa.iloc[3]  # lấy sample đầu tiên

hallu = synthetic(
    row=row,
    client=client,
    model="gpt-4o-mini"
)

print("=== QUESTION ===")
print(row["question"])

print("\n=== CONTEXT ===")
print(row["context"][:1500])

print("\n=== ORIGINAL ANSWER ===")
print(row["gold_answer"])

print("\n=== HALLUCINATED ANSWER ===")
print(hallu)

=== QUESTION ===
Where will failure of the left ventricle cause increased pressure?

=== CONTEXT ===
- (Heart failure) "Backward" failure of the left ventricle causes congestion of the lungs' blood vessels, and therefore causes increased pressure in the lungs. These symptoms are predominantly respiratory in nature.
- (QA_1) Where does failure of the left ventricle cause increased pressure? lungs.
- (QA_2) Where does backwards failure of the left ventricle cause increased pressure? lung's blood vessels.

=== ORIGINAL ANSWER ===
"Backward" failure of the left ventricle causes congestion of the lungs' blood vessels, and therefore causes increased pressure in the lungs. These symptoms are predominantly respiratory in nature.

=== HALLUCINATED ANSWER ===
Failure of the left ventricle causes increased pressure throughout the entire body, leading to widespread swelling and fluid retention in the extremities. This systemic congestion can result in symptoms affecting not just the lungs but also

## 5. Batch generation with checkpoint resume

In [104]:
asqa = asqa.reset_index(drop=True)

In [105]:
def load_checkpoint(checkpoint_path: str) -> pd.DataFrame:
    if checkpoint_path and os.path.exists(checkpoint_path):
        return pd.read_csv(checkpoint_path, dtype={"id": str})
    return pd.DataFrame(columns=["id", "hallu_answer"])


def save_checkpoint(rows: List[Dict[str, str]], checkpoint_path: str) -> None:
    ckpt = pd.DataFrame(rows)
    ckpt.to_csv(checkpoint_path, index=False, encoding="utf-8-sig")


def synthetic_batch(
    df: pd.DataFrame,
    client: OpenAI,
    model: str = MODEL_NAME,
    checkpoint_path: Optional[str] = None,
    checkpoint_freq: int = 25,
) -> pd.DataFrame:
    df = df.copy()
    df["id"] = df["id"].astype(str)

    checkpoint_df = load_checkpoint(checkpoint_path) if checkpoint_path else pd.DataFrame(columns=["id", "hallu_answer"])
    done_ids = set(checkpoint_df["id"].astype(str).tolist())

    rows = checkpoint_df.to_dict("records")

    pbar = tqdm(total=len(df), initial=len(done_ids), desc="Generating hallucinations", unit="sample")

    for _, row in df.iterrows():
        sample_id = str(row["id"])

        if sample_id in done_ids:
            continue

        hallucinated = synthetic(row=row, client=client, model=model)
        if hallucinated is None:
            hallucinated = ""

        rows.append({
            "id": sample_id,
            "hallu_answer": hallucinated,
        })
        done_ids.add(sample_id)

        pbar.update(1)

        if checkpoint_path and len(rows) % checkpoint_freq == 0:
            save_checkpoint(rows, checkpoint_path)
            print(f"💾 checkpoint saved: {len(rows)}/{len(df)}")

        time.sleep(SLEEP_TIME)

    pbar.close()

    if checkpoint_path:
        save_checkpoint(rows, checkpoint_path)
        print(f"✅ final checkpoint saved: {checkpoint_path}")

    return pd.DataFrame(rows)


In [106]:
def build_hallu_dataset(
    base_df: pd.DataFrame,
    client: OpenAI,
    model: str = MODEL_NAME,
    checkpoint_path: str = None,
    hallu_path: str = None,
    labeled_path: str = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    base_df = base_df.copy()
    base_df["id"] = base_df["id"].astype(str)

    # 1. Generate hallucinated answers
    ckpt = synthetic_batch(
        base_df,
        client=client,
        model=model,
        checkpoint_path=checkpoint_path,
        checkpoint_freq=25,
    )

    ckpt["id"] = ckpt["id"].astype(str)

    # 2. Merge back with original samples
    hallu_df = base_df.merge(ckpt, on="id", how="left")

    # 3. Filter empty generations
    empty_mask = hallu_df["hallu_answer"].fillna("").str.len() < 5
    n_empty = int(empty_mask.sum())
    print(f"Empty/invalid hallucinations: {n_empty}")

    if n_empty > 0:
        print("Retrying empty/invalid generations once...")
        retry_df = hallu_df.loc[empty_mask, base_df.columns].copy()
        retry_rows = []

        for _, row in tqdm(retry_df.iterrows(), total=len(retry_df), desc="Retrying"):
            hallucinated = synthetic(row=row, client=client, model=model)
            retry_rows.append({
                "id": str(row["id"]),
                "hallu_answer": hallucinated or "",
            })
            time.sleep(SLEEP_TIME)

        retry_ckpt = pd.DataFrame(retry_rows)
        retry_map = dict(zip(retry_ckpt["id"].astype(str), retry_ckpt["hallu_answer"]))

        hallu_df.loc[empty_mask, "hallu_answer"] = hallu_df.loc[empty_mask, "id"].map(retry_map)

    hallu_df = hallu_df[
        hallu_df["hallu_answer"].notnull() &
        (hallu_df["hallu_answer"].str.len() >= 5)
    ].copy()

    # 4. Export hallu dataset with same schema as gold dataset + answer field
    hallu_export = hallu_df[["id", "question", "context", "gold_answer", "hallu_answer"]].copy()
    if hallu_path:
        hallu_export.to_csv(hallu_path, index=False, encoding="utf-8-sig")
        print(f"Saved hallucinated dataset: {hallu_path}")

    # 5. Build labeled dataset
    # label = 1: supported/gold answer
    # label = 0: hallucinated/unsupported answer
    positive = base_df[["id", "question", "context", "gold_answer"]].copy()
    positive = positive.rename(columns={"gold_answer": "answer"})
    positive["label"] = 1

    negative = hallu_export[["id", "question", "context", "hallu_answer"]].copy()
    negative = negative.rename(columns={"hallu_answer": "answer"})
    negative["id"] = negative["id"].astype(str) + "_hallu"
    negative["label"] = 0

    labeled = pd.concat([positive, negative], ignore_index=True)
    if labeled_path:
        labeled.to_csv(labeled_path, index=False, encoding="utf-8-sig")
        print(f"Saved labeled dataset: {labeled_path}")
    print(labeled["label"].value_counts())

    return labeled, hallu_export

In [107]:
asqa_final, asqa_hallu = build_hallu_dataset(
    base_df=asqa,
    client=client,
    model=MODEL_NAME,
    checkpoint_path=ASQA_CHECKPOINT_PATH,
    hallu_path=ASQA_HALLU_PATH,
    labeled_path=ASQA_LABELED_PATH
)

asqa_final.head()

Generating hallucinations: 100%|██████████| 4353/4353 [00:00<?, ?sample/s]


✅ final checkpoint saved: ../results/synthetic-data/asqa_checkpoint.csv
Empty/invalid hallucinations: 0
Saved hallucinated dataset: ../data/asqa/hallu_asqa.csv
Saved labeled dataset: ../data/asqa/labeled_asqa.csv
label
1    4353
0    4353
Name: count, dtype: int64


,id,question,context,answer,label
0,asqa_0,When does the new bunk'd come out?,- (List of Bunk'd episodes) The new bunk'd epi...,The new bunk'd episode 41 comes out on April 2...,1
1,asqa_1,Who won the 2016 ncaa football national champi...,- (2016 College Football Playoff National Cham...,The 2015 - 2016 season's ncaa national footbal...,1
2,asqa_2,When was the last time the death penalty was u...,"- (QA_1) As of 2017, when was the last time th...",The last time the death penalty was used in pa...,1
3,asqa_3,Where will failure of the left ventricle cause...,"- (Heart failure) ""Backward"" failure of the le...","""Backward"" failure of the left ventricle cause...",1
4,asqa_4,Who won the war between ethiopia and italy?,- (Second Italo-Ethiopian War) The first war b...,The first war between Italy and Ethiopia took ...,1


In [108]:
labeled_msmacro, hallu_msmacro = build_hallu_dataset(
    msmarco_500_export,
    client=client,
    model=MODEL_NAME,
    checkpoint_path=MS_CHECKPOINT_PATH,
    hallu_path=MS_HALLU_PATH,
    labeled_path=MS_LABELED_PATH,
)

labeled_msmacro.head()


Generating hallucinations: 100%|██████████| 500/500 [00:00<?, ?sample/s]

✅ final checkpoint saved: ../results/synthetic-data/msmarco_checkpoint.csv
Empty/invalid hallucinations: 0
Saved hallucinated dataset: ../data/ms-marco/hallu_msmarco.csv
Saved labeled dataset: ../data/ms-marco/labeled_msmarco.csv
label
1    500
0    500
Name: count, dtype: int64


,id,question,context,answer,label
0,10736,buying a car dealer vs. private seller,[P1] Source: http://www.consumerhelp.ie/buying...,Traders are often called “dealers” and sell ca...,1
1,12719,convert dollar pay into salary to twice a month,[P1] Source: http://www.ehow.com/how_12044390_...,Determine what your paydays will be throughout...,1
2,17806,how to plant dwarf apple trees,[P1] Source: http://www.gardenguides.com/79766...,Locate a site for the dwarf apple tree where i...,1
3,14675,what arthritis is,[P1] Source: http://www.healthline.com/health/...,Arthritis is inflammation of the joints (the p...,1
4,16014,how to record audio with iphone,[P1] Source: http://www.solveyourtech.com/reco...,1: Open the Voice Memos app. If you cannot fin...,1


## 5. Wiki Eval split`ing

In [109]:
def _parse_context(context: List[str]) -> str:
    return " ".join(ctx.strip() for ctx in context)

In [110]:
def convert_wikieval_to_faithfulness_dataset(ds) -> pd.DataFrame:
    records = []
    for i, item in tqdm(enumerate(ds), total=len(ds)):
        question = item["question"].replace("Question: ", "").strip()
        context = _parse_context(item["context_v1"])
        gold_answer = item["answer"].replace("Answer: ", "").strip()
        bad_answer = item["poor_answer"]

        # Positive example
        records.append({
            "id": f"{i}_pos",
            "question": question,
            "context": context,
            "answer": gold_answer,
            "label": 1
        })

        # Negative example
        records.append({
            "id": f"{i}_neg",
            "question": question,
            "context": context,
            "answer": bad_answer,
            "label": 0
        })

    df = pd.DataFrame(records)
    return df


In [111]:
labeled_df = convert_wikieval_to_faithfulness_dataset(wiki)
print("Labeled WikiEval dataset:")
print(labeled_df.shape)
display(labeled_df.head(3))

100%|██████████| 50/50 [00:00<00:00, 1050.37it/s]

Labeled WikiEval dataset:
(100, 5)


,id,question,context,answer,label
0,0_pos,When is the scheduled launch date and time for...,The PSLV-C56 is the 58th mission of Indian Spa...,The PSLV-C56 mission is scheduled to be launch...,1
1,0_neg,When is the scheduled launch date and time for...,The PSLV-C56 is the 58th mission of Indian Spa...,The scheduled launch date and time for the PSL...,0
2,1_pos,What is the objective of the Uzbekistan-Afghan...,The Uzbekistan–Afghanistan–Pakistan Railway Pr...,The objective of the Uzbekistan-Afghanistan-Pa...,1


In [112]:
def create_dev(ds) -> pd.DataFrame:
    records = []
    for i, item in tqdm(enumerate(ds), total=len(ds)):
        question = item["question"].replace("Question: ", "").strip()
        context = _parse_context(item["context_v1"])
        gold_answer = item["answer"].replace("Answer: ", "").strip()
        predicted_answer = item["ungrounded_answer"].replace("Answer: ", "").strip()

        records.append({
            "id": f"{i}",
            "question": question,
            "context": context,
            "gold_answer": gold_answer,
            "predicted_answer": predicted_answer
        })

    df = pd.DataFrame(records)
    return df
    

## 6. Final dataset overview

In [113]:
raw_df = pd.DataFrame(wiki)
raw_df.to_csv(WIKI_PATH, index=False)
labeled_df.to_csv(WIKI_LABELED_PATH, index=False)
dev_df = create_dev(wiki)
dev_df.to_csv(WIKI_DEV_PATH, index=False)

print(f"Saved original WikiEval dataset to {WIKI_PATH}")
print(f"Saved labeled WikiEval dataset to {WIKI_LABELED_PATH}") 
print(f"Saved dev set to {WIKI_DEV_PATH}")

100%|██████████| 50/50 [00:00<00:00, 10017.92it/s]

Saved original WikiEval dataset to ../data/wikiEval/wikieval.csv
Saved labeled WikiEval dataset to ../data/wikiEval/labeled_wikieval.csv
Saved dev set to ../data/wikiEval/dev_wikieval.csv


In [114]:
print(f"Save ASQA hallucination dataset to {ASQA_HALLU_PATH}")
print(f"Save ASQA final dataset to {ASQA_LABELED_PATH}")


print(f"Save MS MARCO hallucination dataset to {MS_HALLU_PATH}")
print(f"Save MS MARCO final dataset to {MS_LABELED_PATH}")

Save ASQA hallucination dataset to ../data/asqa/hallu_asqa.csv
Save ASQA final dataset to ../data/asqa/labeled_asqa.csv
Save MS MARCO hallucination dataset to ../data/ms-marco/hallu_msmarco.csv
Save MS MARCO final dataset to ../data/ms-marco/labeled_msmarco.csv


## Merge version:
Combine all 3 labeled datasets:
- ASQA labeled
- MS MARCO labeled
- WikiEval labeled

Add a column "dataset" to indicate the origin of each sample.

In [115]:
# ## Merge version:
# Combine all 3 labeled datasets:
# - ASQA labeled
# - MS MARCO labeled
# - WikiEval labeled

# Add a column "dataset" to indicate the origin of each sample.

asqa_labeled = pd.read_csv(ASQA_LABELED_PATH)
asqa_labeled["dataset"] = "asqa"
ms_labeled = pd.read_csv(MS_LABELED_PATH)
ms_labeled["dataset"] = "msmarco"
wiki_labeled = pd.read_csv(WIKI_LABELED_PATH)
wiki_labeled["dataset"] = "wikieval"

merged_labeled = pd.concat([asqa_labeled, ms_labeled, wiki_labeled], ignore_index=True)
merged_labeled.to_csv("../data/labeled_merged.csv", index=False, encoding="utf-8-sig")

### merge ragas checkpoint for ASQA, MS MARCO, WikiEval

In [116]:
ASQA_RAGAS_CP = "../results/ragas_filter/training/asqa_ragas_checkpoints.csv"
MS_RAGAS_CP = "../results/ragas_filter/classification/msmarco_ragas_checkpoints.csv"
WIKI_RAGAS_CP = "../results/ragas_filter/classification/wikieval_ragas_checkpoints.csv"

NEW_CHECKPOINT = "../results/ragas_filter/merged/merged_ragas_checkpoints.csv"

asqa_ragas = pd.read_csv(ASQA_RAGAS_CP)
asqa_ragas["dataset"] = "asqa"
# resume the 'sample_idx" column in asqa_ragas to be continuous with ms_ragas and wiki_ragas
last_idx = asqa_ragas["sample_idx"].max()
ms_ragas = pd.read_csv(MS_RAGAS_CP)
ms_ragas["dataset"] = "msmarco"
ms_ragas["sample_idx"] = ms_ragas["sample_idx"] + last_idx
last_idx = ms_ragas["sample_idx"].max()
wiki_ragas = pd.read_csv(WIKI_RAGAS_CP)
wiki_ragas["dataset"] = "wikieval"
wiki_ragas["sample_idx"] = wiki_ragas["sample_idx"] + last_idx

if not os.path.exists("../results/ragas_filter/merged"):
    os.makedirs("../results/ragas_filter/merged")


merged_ragas = pd.concat([asqa_ragas, ms_ragas, wiki_ragas], ignore_index=True)
merged_ragas.to_csv(NEW_CHECKPOINT, index=False, encoding="utf-8-sig")

In [ ]:
display(merged_labeled.head())

print(merged_labeled.describe())

# thống kê số lượng sample theo dataset "dataset", "accept", "reject", "total"
df_stats = merged_labeled.groupby("dataset")["label"].value_counts().unstack(fill_value=0)
df_stats["total"] = df_stats.sum(axis=1)
df_stats = df_stats.rename(columns={0: "reject", 1: "accept"})

display(df_stats)